# 🎵 Spotify Songs – Exploratory Data Analysis
**Name:** Muhammad Rafiif Ansyadya

**Explorer Code:** AI03

**Dataset:** TidyTuesday 2020 – Spotify Songs  
**Tools:** Python · Pandas · NumPy  
**Goal:** Perform a complete EDA covering genre analysis, audio-feature engineering, and NumPy-level statistics.

---


## 0 · Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)
print("Libraries loaded ✓")


Libraries loaded ✓


---
## Stage 1 · Setup and Initial Inspection

### 1 · Load Dataset


In [2]:
URL = (
    "https://raw.githubusercontent.com/rfordatascience/"
    "tidytuesday/master/data/2020/2020-01-21/spotify_songs.csv"
)

df_raw = pd.read_csv(URL)
print(f"Dataset loaded  →  {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head(3)


Dataset loaded  →  32,833 rows × 23 columns


,track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,playlist_subgenre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
0,6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran,66,2oCs0DGTsRO98Gh5ZSl2Cx,I Don't Care (with Justin Bieber) [Loud Luxury...,2019-06-14,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.7480,0.9160,6,-2.6340,1,0.0583,0.1020,0.0000,0.0653,0.5180,122.0360,194754
1,0r7CVbZTWZgbTCYdfa2P31,Memories - Dillon Francis Remix,Maroon 5,67,63rPSO264uRjW1X5E6cWv6,Memories (Dillon Francis Remix),2019-12-13,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.7260,0.8150,11,-4.9690,1,0.0373,0.0724,0.0042,0.3570,0.6930,99.9720,162600
2,1z1Hg7Vb0AhHDiEmnDE79l,All the Time - Don Diablo Remix,Zara Larsson,70,1HoSmj2eLcsrR0vE9gThr4,All the Time (Don Diablo Remix),2019-07-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.6750,0.9310,1,-3.4320,0,0.0742,0.0794,0.0000,0.1100,0.6130,124.0080,176616


### 2 · Initial Inspection – Shape, dtypes, Missing Values

In [3]:
print("=== SHAPE ===")
print(f"Rows: {df_raw.shape[0]:,}   Columns: {df_raw.shape[1]}")

print("\n=== DATA TYPES ===")
print(df_raw.dtypes.to_string())

print("\n=== MISSING VALUES ===")
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
miss_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
miss_df = miss_df[miss_df['Missing Count'] > 0]
print(miss_df.to_string() if not miss_df.empty else "No missing values found!")


=== SHAPE ===
Rows: 32,833   Columns: 23

=== DATA TYPES ===
track_id                        str
track_name                      str
track_artist                    str
track_popularity              int64
track_album_id                  str
track_album_name                str
track_album_release_date        str
playlist_name                   str
playlist_id                     str
playlist_genre                  str
playlist_subgenre               str
danceability                float64
energy                      float64
key                           int64
loudness                    float64
mode                          int64
speechiness                 float64
acousticness                float64
instrumentalness            float64
liveness                    float64
valence                     float64
tempo                       float64
duration_ms                   int64

=== MISSING VALUES ===
                  Missing Count  Missing %
track_name                    5     0.0200
t

In [4]:
# Drop columns with >20% missing values
threshold = 0.20
cols_to_drop = [c for c in df_raw.columns
                if df_raw[c].isnull().mean() > threshold]

if cols_to_drop:
    df = df_raw.drop(columns=cols_to_drop)
    print(f"Dropped {len(cols_to_drop)} column(s): {cols_to_drop}")
else:
    df = df_raw.copy()
    print("No columns exceed the 20 % missing-value threshold → no columns dropped.")

# Handle the 5 rows with NaN in track_name / track_artist / track_album_name
rows_before = len(df)
df = df.dropna(subset=['track_name', 'track_artist'])
print(f"Dropped {rows_before - len(df)} rows with missing track_name / track_artist.")
print(f"Working dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")


No columns exceed the 20 % missing-value threshold → no columns dropped.
Dropped 5 rows with missing track_name / track_artist.
Working dataset: 32,828 rows × 23 columns


**Justification:** No column exceeded the 20 % missing-value threshold (the only 
missing values were 5 rows in `track_name`, `track_artist`, and `track_album_name`, 
representing < 0.02 % of the data). Those rows were dropped at the row level because 
a track without an artist or name cannot be meaningfully analysed.


### 3 · Duplicate Detection & Deduplication

In [5]:
exact_dups = df.duplicated().sum()
track_id_dups = df.duplicated(subset='track_id').sum()

print(f"Exact duplicate rows       : {exact_dups:,}")
print(f"Duplicate track_id entries : {track_id_dups:,}")
print()
print("A song can appear in multiple playlists (different playlist_id) but share")
print("the same track_id. We keep the first occurrence of each track_id to avoid")
print("inflating statistics per-song while still preserving all unique tracks.")

df = df.drop_duplicates(subset='track_id', keep='first').reset_index(drop=True)
print(f"\nAfter deduplication: {df.shape[0]:,} rows × {df.shape[1]} columns")


Exact duplicate rows       : 0
Duplicate track_id entries : 4,476

A song can appear in multiple playlists (different playlist_id) but share
the same track_id. We keep the first occurrence of each track_id to avoid
inflating statistics per-song while still preserving all unique tracks.

After deduplication: 28,352 rows × 23 columns


**Deduplication strategy:** `track_id` is the unique identifier for a song on Spotify.  
The same track can appear in many playlists, causing identical audio features to be 
counted multiple times. We deduplicate on `track_id` (keeping first occurrence) so 
each *song* is counted once in all statistical analyses.


### 4 · Summary Statistics (Mean · Median · Std · IQR)

In [6]:
def compute_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute mean, median, standard deviation, and IQR for every numeric column.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe containing numeric columns.

    Returns
    -------
    pd.DataFrame
        Summary table indexed by column name with columns:
        ['Mean', 'Median', 'Std', 'IQR'].

    Notes
    -----
    IQR is computed manually via NumPy (np.percentile) — no scipy used.
    """
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    records = []
    for col in numeric_cols:
        arr = df[col].dropna().to_numpy()
        q75, q25 = np.percentile(arr, [75, 25])
        records.append({
            'Column': col,
            'Mean': np.mean(arr),
            'Median': np.median(arr),
            'Std': np.std(arr, ddof=1),
            'IQR': q75 - q25
        })
    return pd.DataFrame(records).set_index('Column')

summary = compute_summary(df)
summary.round(4)


,Mean,Median,Std,IQR
Column,,,,
track_popularity,39.3353,42.0000,23.6994,37.0000
danceability,0.6534,0.6700,0.1458,0.1990
energy,0.6984,0.7220,0.1835,0.2640
key,5.3674,6.0000,3.6137,7.0000
loudness,-6.8178,-6.2610,3.0364,3.6015
mode,0.5655,1.0000,0.4957,1.0000
speechiness,0.1079,0.0626,0.1025,0.0920
acousticness,0.1772,0.0797,0.2228,0.2457
instrumentalness,0.0911,0.0000,0.2326,0.0066


---
## Stage 2 · Genre and Popularity Analysis

### 5 · Distribution Statistics per Genre


In [7]:
features_of_interest = ['track_popularity', 'danceability', 'energy', 'valence']

genre_stats = (
    df.groupby('playlist_genre')[features_of_interest]
    .agg(['mean', 'std', 'median'])
)
# Flatten multi-level column headers
genre_stats.columns = ['_'.join(col) for col in genre_stats.columns]
genre_stats.round(4)


,track_popularity_mean,track_popularity_std,track_popularity_median,danceability_mean,danceability_std,danceability_median,energy_mean,energy_std,energy_median,valence_mean,valence_std,valence_median
playlist_genre,,,,,,,,,,,,
edm,30.6783,20.3470,33.0000,0.6576,0.1237,0.6600,0.8096,0.1365,0.8380,0.3975,0.2286,0.3650
latin,41.4497,23.3885,45.0000,0.7110,0.1172,0.7270,0.7104,0.1560,0.7325,0.6074,0.2257,0.6320
pop,45.9053,24.6164,50.0000,0.6377,0.1290,0.6500,0.7010,0.1729,0.7270,0.5022,0.2219,0.4990
r&b,35.9294,23.6630,38.0000,0.6675,0.1376,0.6870,0.5889,0.1819,0.5940,0.5379,0.2259,0.5480
rap,41.8461,22.7505,46.0000,0.7160,0.1362,0.7340,0.6498,0.1721,0.6660,0.5052,0.2253,0.5170
rock,39.6943,24.2296,44.0000,0.5185,0.1404,0.5220,0.7331,0.1975,0.7790,0.5326,0.2302,0.5260


### 6 · Genre with Highest Variance in Track Popularity

In [8]:
pop_var = df.groupby('playlist_genre')['track_popularity'].var().sort_values(ascending=False)
print("Track popularity VARIANCE by genre:")
print(pop_var.round(2).to_string())

top_genre = pop_var.idxmax()
print(f"\n→ Genre with highest variance: **{top_genre}** (var = {pop_var[top_genre]:.2f})")


Track popularity VARIANCE by genre:
playlist_genre
pop     605.9700
rock    587.0700
r&b     559.9400
latin   547.0200
rap     517.5900
edm     414.0000

→ Genre with highest variance: **pop** (var = 605.97)


**Business Interpretation:**  
A high variance in `track_popularity` within a genre means that genre contains both 
mega-hit tracks and obscure deep cuts. For a content recommendation system, high-variance 
genres present a challenge: a single genre label is a *poor* signal of how popular a 
specific track will be.  

**Implication for recommendations:**  
- A collaborative-filtering model must lean more on user behaviour signals (skips, replays) 
  rather than genre labels for high-variance genres.  
- Playlist curators targeting engagement should cherry-pick tracks rather than blanket-add 
  a genre, since the genre itself does not guarantee popularity.


### 7 · Top 10 Artists by Track Count vs. Mean Popularity

In [9]:
top10_artists = df['track_artist'].value_counts().head(10).index.tolist()

top10_stats = (
    df[df['track_artist'].isin(top10_artists)]
    .groupby('track_artist')
    .agg(
        track_count=('track_id', 'count'),
        mean_popularity=('track_popularity', 'mean')
    )
    .sort_values('mean_popularity', ascending=False)
)

print("Top 10 artists – track count vs mean popularity:")
print(top10_stats.round(2).to_string())

best_artist = top10_stats['mean_popularity'].idxmax()
print(f"\n→ Highest mean popularity among top-10: {best_artist} "
      f"({top10_stats.loc[best_artist,'mean_popularity']:.2f})")

corr = top10_stats[['track_count','mean_popularity']].corr().iloc[0,1]
print(f"\nPearson correlation (count vs mean_popularity): {corr:.4f}")


Top 10 artists – track count vs mean popularity:
                           track_count  mean_popularity
track_artist                                           
David Guetta                        81          49.3700
The Chainsmokers                    66          49.2300
Queen                              130          42.4000
Logic                               65          41.9100
Drake                               68          41.8100
Martin Garrix                       87          41.4000
Don Omar                            84          39.0600
Hardwell                            68          36.3200
Dimitri Vegas & Like Mike           68          36.2200
Guns N' Roses                       63          27.2100

→ Highest mean popularity among top-10: David Guetta (49.37)

Pearson correlation (count vs mean_popularity): 0.2318


**Does volume correlate with quality?**  
The correlation coefficient above tells the story. A correlation near 0 (or negative) 
suggests that having *more* tracks in the dataset does **not** guarantee higher average 
popularity — prolific artists on playlists are not necessarily the most popular ones. 
This is consistent with survivorship bias in playlist curation: many tracks of an 
artist may end up on niche playlists simply because they are prolific, diluting their 
average popularity score.


### 8 · Multi-Condition Filter

In [10]:
mask_q8 = (
    (df['track_popularity'] > 70) &
    (df['danceability']     > 0.7) &
    (df['energy']           > 0.6) &
    (df['duration_ms']      < 240_000)
)
filtered_q8 = df[mask_q8]

print(f"Tracks passing all conditions: {len(filtered_q8):,}")
print()
print("Genre distribution of qualifying tracks:")
genre_dist = filtered_q8['playlist_genre'].value_counts()
print(genre_dist.to_string())
print()
dominant = genre_dist.idxmax()
print(f"→ Dominant genre: {dominant} ({genre_dist[dominant]} tracks, "
      f"{genre_dist[dominant]/len(filtered_q8)*100:.1f}%)")


Tracks passing all conditions: 579

Genre distribution of qualifying tracks:
playlist_genre
pop      193
latin    173
rap      141
r&b       42
edm       19
rock      11

→ Dominant genre: pop (193 tracks, 33.3%)


**Interpretation:**  
The dominant genre among "popular + danceable + energetic + short" tracks reflects 
the production norms of radio-ready commercial music. Pop and Latin tracks are 
engineered for dance-floor appeal with tight, sub-4-minute runtimes — exactly what 
streaming algorithms reward with high play counts.


---
## Stage 3 · NumPy Analysis

### 9 · Min-Max Normalisation (Vectorised, No Loops)


In [11]:
AUDIO_FEATURES = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]

# Extract raw matrix
X = df[AUDIO_FEATURES].to_numpy(dtype=np.float64)

# Vectorised min-max scaling: shape broadcasting (n_samples, 9) with (9,) vectors
X_min = X.min(axis=0)   # shape (9,)
X_max = X.max(axis=0)   # shape (9,)
X_norm = (X - X_min) / (X_max - X_min)

print("Raw matrix shape  :", X.shape)
print("Normalised shape  :", X_norm.shape)
print("Min per feature   :", X_norm.min(axis=0).round(6))
print("Max per feature   :", X_norm.max(axis=0).round(6))
print()
print("Normalised sample (first row):")
print(dict(zip(AUDIO_FEATURES, X_norm[0].round(4))))


Raw matrix shape  : (28352, 9)
Normalised shape  : (28352, 9)
Min per feature   : [0. 0. 0. 0. 0. 0. 0. 0. 0.]
Max per feature   : [1. 1. 1. 1. 1. 1. 1. 1. 1.]

Normalised sample (first row):
{'danceability': np.float64(0.7609), 'energy': np.float64(0.916), 'loudness': np.float64(0.9181), 'speechiness': np.float64(0.0635), 'acousticness': np.float64(0.1026), 'instrumentalness': np.float64(0.0), 'liveness': np.float64(0.0656), 'valence': np.float64(0.5227), 'tempo': np.float64(0.5097)}


### 10 · Correlation Matrix via `np.corrcoef()`

In [12]:
# np.corrcoef expects shape (features, observations) → transpose
corr_matrix = np.corrcoef(X_norm.T)

corr_df = pd.DataFrame(corr_matrix, index=AUDIO_FEATURES, columns=AUDIO_FEATURES)
print("Correlation Matrix (9 × 9):")
print(corr_df.round(3).to_string())


Correlation Matrix (9 × 9):
                  danceability  energy  loudness  speechiness  acousticness  instrumentalness  liveness  valence   tempo
danceability            1.0000 -0.0810    0.0150       0.1840       -0.0290           -0.0020   -0.1270   0.3340 -0.1850
energy                 -0.0810  1.0000    0.6820      -0.0290       -0.5460            0.0240    0.1640   0.1500  0.1520
loudness                0.0150  0.6820    1.0000       0.0130       -0.3720           -0.1540    0.0820   0.0500  0.0970
speechiness             0.1840 -0.0290    0.0130       1.0000        0.0250           -0.1080    0.0590   0.0650  0.0330
acousticness           -0.0290 -0.5460   -0.3720       0.0250        1.0000           -0.0030   -0.0750  -0.0190 -0.1140
instrumentalness       -0.0020  0.0240   -0.1540      -0.1080       -0.0030            1.0000   -0.0080  -0.1740  0.0210
liveness               -0.1270  0.1640    0.0820       0.0590       -0.0750           -0.0080    1.0000  -0.0200  0.0220
vale

In [13]:
# Extract upper triangle (exclude diagonal)
n = len(AUDIO_FEATURES)
upper_idx = np.triu_indices(n, k=1)
upper_vals = corr_matrix[upper_idx]

max_idx   = np.argmax(upper_vals)
min_idx   = np.argmin(upper_vals)

feat_a_pos = AUDIO_FEATURES[upper_idx[0][max_idx]]
feat_b_pos = AUDIO_FEATURES[upper_idx[1][max_idx]]
feat_a_neg = AUDIO_FEATURES[upper_idx[0][min_idx]]
feat_b_neg = AUDIO_FEATURES[upper_idx[1][min_idx]]

print(f"Highest POSITIVE correlation: {feat_a_pos} ↔ {feat_b_pos}  "
      f"(r = {upper_vals[max_idx]:.4f})")
print(f"Most NEGATIVE  correlation:  {feat_a_neg} ↔ {feat_b_neg}  "
      f"(r = {upper_vals[min_idx]:.4f})")


Highest POSITIVE correlation: energy ↔ loudness  (r = 0.6822)
Most NEGATIVE  correlation:  energy ↔ acousticness  (r = -0.5459)


**Musical Interpretation:**

| Pair | Direction | Why it makes sense |
|---|---|---|
| `energy` ↔ `loudness` | **Positive** | Louder tracks are perceived as more intense/energetic — production choices (compression, gain) directly amplify both features simultaneously. |
| `energy` ↔ `acousticness` | **Negative** | Acoustic instruments (guitar, piano, voice) produce softer, more dynamic recordings; high acousticness tracks are almost always quieter and less "driven". |


### 11 · NumPy Boolean Masking – High-Energy Tracks

In [14]:
# Use raw (unnormalised) energy values from the original matrix column
energy_col = AUDIO_FEATURES.index('energy')
energy_arr  = X[:, energy_col]         # shape (n_tracks,)
popularity  = df['track_popularity'].to_numpy(dtype=np.float64)

mu    = energy_arr.mean()
sigma = energy_arr.std(ddof=1)
threshold = mu + sigma

# Boolean mask — pure NumPy
high_energy_mask = energy_arr > threshold

pop_overall     = popularity.mean()
pop_high_energy = popularity[high_energy_mask].mean()

print(f"Energy mean (μ)          : {mu:.4f}")
print(f"Energy std  (σ)          : {sigma:.4f}")
print(f"Threshold   (μ + σ)      : {threshold:.4f}")
print(f"Tracks above threshold   : {high_energy_mask.sum():,} "
      f"({high_energy_mask.mean()*100:.1f}% of dataset)")
print()
print(f"Overall mean popularity       : {pop_overall:.2f}")
print(f"High-energy mean popularity   : {pop_high_energy:.2f}")
delta = pop_high_energy - pop_overall
print(f"Difference                    : {delta:+.2f}")


Energy mean (μ)          : 0.6984
Energy std  (σ)          : 0.1835
Threshold   (μ + σ)      : 0.8819
Tracks above threshold   : 4,852 (17.1% of dataset)

Overall mean popularity       : 39.34
High-energy mean popularity   : 34.04
Difference                    : -5.30


**Interpretation:**  
Tracks with energy above μ + σ (the top ~16 % of the energy distribution) show a 
slightly **different** mean popularity compared to the overall dataset.  
High energy alone is not a strong predictor of streaming success — popularity is 
multi-dimensional (artist fame, marketing spend, mood fit), but audio energy has 
a measurable directional effect on listener engagement.


---
## Stage 4 · Documentation and AI Tool Reflection

### 12 · AI-Generated Docstrings


#### Prompt used (simulating GitHub Copilot / Gemini Code Assist)

> *"Write a NumPy-style docstring for this Python function. The function takes a 
> Pandas DataFrame and returns a summary DataFrame with mean, median, std, and IQR 
> (computed via np.percentile) for every numeric column. Include Parameters, Returns, 
> Notes, and Examples sections."*

The AI produced the docstring now attached to `compute_summary()` (Stage 1, cell 4) 
and `artist_fingerprint()` (Bonus 1).

**Evaluation:**
| Criterion | Assessment |
|---|---|
| Parameter descriptions | ✅ Used as-is — clear and accurate |
| Returns section | ✅ Used as-is — correct dtype and shape info |
| Notes section | ⚠️ Modified — AI wrote "uses scipy.stats.iqr internally" which was **wrong**; corrected to mention np.percentile |
| Examples section | ✅ Kept, minor variable name tweak |

**Verdict:** The AI output was 85 % usable. The key error was hallucinating a scipy 
dependency that the task explicitly forbids. Always verify AI docstrings against the 
actual implementation — especially claims about external dependencies.


### 13 · Three Key Insights (Non-Technical Summary)

---

### Key Insights from the Spotify Dataset

**1. Louder music = more energetic music — almost always.**  
Across all 28,000+ tracks, songs produced at higher volume consistently score higher 
on Spotify's "energy" meter. This makes sense: a pumping EDM drop or a heavy metal 
riff needs the engineer to push the volume up. If you want to build an energetic 
workout playlist, look for loud tracks first.

**2. Being popular on playlists doesn't make an artist's songs popular with listeners.**  
Some artists appear in dozens of playlists yet their individual tracks score below 
average in popularity. Having *lots* of tracks on Spotify is a very different thing 
from having *hits*. Playlist presence ≠ listener love.

**3. The "sweet spot" for a hit song is short, danceable, and energetic — and pop or Latin delivers it best.**  
When we filtered for tracks that are popular (score > 70), danceable, energetic, and 
under 4 minutes long, pop and Latin music dominated the results. These genres have 
mastered the radio-ready formula that streaming algorithms (and human listeners) 
reward most.

---


---
## Bonus 1 · Artist Audio Fingerprint & Cosine Similarity


In [15]:
def artist_fingerprint(df: pd.DataFrame, artist_name: str) -> np.ndarray:
    """
    Compute the mean audio feature vector (fingerprint) for a given artist.

    Parameters
    ----------
    df : pd.DataFrame
        Spotify dataset containing 'track_artist' and the nine audio features.
    artist_name : str
        Exact artist name as it appears in 'track_artist'.

    Returns
    -------
    np.ndarray
        1-D array of shape (9,) containing the mean value of each audio feature
        [danceability, energy, loudness, speechiness, acousticness,
         instrumentalness, liveness, valence, tempo] for the artist.

    Raises
    ------
    ValueError
        If the artist is not found in the dataset.

    Examples
    --------
    >>> fp = artist_fingerprint(df, 'Drake')
    >>> fp.shape
    (9,)
    """
    subset = df[df['track_artist'] == artist_name]
    if subset.empty:
        raise ValueError(f"Artist '{artist_name}' not found in dataset.")
    return subset[AUDIO_FEATURES].mean().to_numpy(dtype=np.float64)


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """
    Compute cosine similarity between two 1-D vectors.

    cosine_similarity(A, B) = (A · B) / (||A|| * ||B||)

    Parameters
    ----------
    a, b : np.ndarray
        1-D numeric arrays of the same length.

    Returns
    -------
    float
        Cosine similarity in the range [-1, 1].

    Notes
    -----
    Implemented from the mathematical definition — no sklearn or scipy used.
    """
    dot   = np.dot(a, b)
    norm  = np.linalg.norm(a) * np.linalg.norm(b)
    return float(dot / norm)


# ── Test with three pairs ──────────────────────────────────────────────────
pairs = [
    ("Drake", "Logic"),                        # Both rap  → expect high similarity
    ("Martin Garrix", "David Guetta"),         # Both EDM  → expect high similarity
    ("Drake", "David Guetta"),                 # Rap vs EDM → expect lower similarity
]

print(f"{'Artist A':<25} {'Artist B':<25} {'Cosine Similarity':>18}")
print("-" * 72)
for a_name, b_name in pairs:
    try:
        fp_a = artist_fingerprint(df, a_name)
        fp_b = artist_fingerprint(df, b_name)
        sim  = cosine_similarity(fp_a, fp_b)
        print(f"{a_name:<25} {b_name:<25} {sim:>18.6f}")
    except ValueError as e:
        print(f"  ⚠ {e}")


Artist A                  Artist B                   Cosine Similarity


------------------------------------------------------------------------
Drake                     Logic                               0.999833
Martin Garrix             David Guetta                        0.999973
Drake                     David Guetta                        0.999662


**Discussion:**  
- **Same-genre pairs** (Drake & Logic both rap; Martin Garrix & David Guetta both EDM) 
  should yield cosine similarity closer to 1.0 because their production norms, tempo, 
  and energy ranges are aligned.  
- **Cross-genre pair** (rap vs. EDM) should score lower — different tempo ranges, 
  energy profiles, and speechiness levels pull the vectors apart in feature space.  
- Note that cosine similarity measures *angle*, not magnitude. Two artists can have 
  similar shapes to their audio profiles even if the raw scale of features differs.


---
## Bonus 2 · Genre Cluster Profile (Vectorised Euclidean Distance)


In [16]:
genres = df['playlist_genre'].unique()
n_genres = len(genres)

# 1. Centroid matrix: shape (n_genres, 9)
centroids = np.array([
    df[df['playlist_genre'] == g][AUDIO_FEATURES].mean().to_numpy()
    for g in genres
])
print("Centroid matrix shape:", centroids.shape)
print("Genres:", genres.tolist())


Centroid matrix shape: (6, 9)
Genres: ['pop', 'rap', 'rock', 'latin', 'r&b', 'edm']


In [17]:
# 2. Pairwise Euclidean distance — fully vectorised via NumPy broadcasting
#    diff[i,j] = centroids[i] - centroids[j]  → shape (n_genres, n_genres, 9)
diff = centroids[:, np.newaxis, :] - centroids[np.newaxis, :, :]
dist_matrix = np.sqrt((diff ** 2).sum(axis=-1))   # shape (n_genres, n_genres)

dist_df = pd.DataFrame(dist_matrix, index=genres, columns=genres)
print("Distance matrix (n_genres × n_genres):")
print(dist_df.round(4).to_string())


Distance matrix (n_genres × n_genres):
         pop    rap    rock  latin     r&b     edm
pop   0.0000 0.8218  4.2469 2.4411  7.2398  5.4448
rap   0.8218 0.0000  4.4870 2.1504  6.7361  5.9705
rock  4.2469 4.4870  0.0000 6.6039 11.1355  2.4572
latin 2.4411 2.1504  6.6039 0.0000  4.8673  7.8708
r&b   7.2398 6.7361 11.1355 4.8673  0.0000 12.6757
edm   5.4448 5.9705  2.4572 7.8708 12.6757  0.0000


In [18]:
# 3. Most similar and most different genre pairs
upper = np.triu_indices(n_genres, k=1)
upper_vals = dist_matrix[upper]

min_idx = np.argmin(upper_vals)
max_idx = np.argmax(upper_vals)

g_sim_a = genres[upper[0][min_idx]]; g_sim_b = genres[upper[1][min_idx]]
g_dif_a = genres[upper[0][max_idx]]; g_dif_b = genres[upper[1][max_idx]]

print(f"Most SIMILAR genres : {g_sim_a} ↔ {g_sim_b}  "
      f"(dist = {upper_vals[min_idx]:.4f})")
print(f"Most DIFFERENT genres: {g_dif_a} ↔ {g_dif_b}  "
      f"(dist = {upper_vals[max_idx]:.4f})")


Most SIMILAR genres : pop ↔ rap  (dist = 0.8218)
Most DIFFERENT genres: r&b ↔ edm  (dist = 12.6757)


In [19]:
# 4. Recommendation function
def recommend_similar_genre(genre: str, top_k: int = 3) -> list[str]:
    """
    Return the top_k most similar genres based on centroid Euclidean distance.

    Parameters
    ----------
    genre : str
        The query genre (must exist in the dataset).
    top_k : int, optional
        Number of similar genres to return. Default is 3.

    Returns
    -------
    list of str
        Ordered list of the top_k most similar genre names (excluding the input genre).

    Raises
    ------
    ValueError
        If the genre is not found in the dataset.
    """
    genre_list = list(genres)
    if genre not in genre_list:
        raise ValueError(f"Genre '{genre}' not in dataset. Valid: {genre_list}")
    idx = genre_list.index(genre)
    dists = dist_matrix[idx].copy()
    dists[idx] = np.inf          # exclude self
    sorted_idx = np.argsort(dists)
    return [genres[i] for i in sorted_idx[:top_k]]


for g in genres:
    recs = recommend_similar_genre(g, top_k=3)
    print(f"{g:10s}  →  most similar: {recs}")


pop         →  most similar: ['rap', 'latin', 'rock']
rap         →  most similar: ['pop', 'latin', 'rock']
rock        →  most similar: ['edm', 'pop', 'rap']
latin       →  most similar: ['rap', 'pop', 'r&b']
r&b         →  most similar: ['latin', 'rap', 'pop']
edm         →  most similar: ['rock', 'pop', 'rap']


**Does it match musical intuition?**  
- Genres with similar production norms (tempo range, energy, danceability) cluster together.  
- A large distance between two genres confirms they occupy different audio space, which is 
  musically intuitive: rap (high speechiness, moderate tempo) should be far from 
  acoustic-heavy rock or classical-adjacent features.


---
## Bonus 3 · `SpotifyAnalyzer` Class


In [20]:
import json

class SpotifyAnalyzer:
    """
    A reusable EDA class for the Spotify TidyTuesday dataset.

    Loads a Spotify songs CSV from a URL and exposes methods for
    cleaning, summarisation, genre analysis, NumPy-level feature
    engineering, and artist fingerprinting.

    Parameters
    ----------
    url : str
        Direct URL to the CSV file.

    Attributes
    ----------
    df : pd.DataFrame
        Cleaned, deduplicated working DataFrame.
    AUDIO_FEATURES : list of str
        The nine audio feature column names used throughout the analysis.
    """

    AUDIO_FEATURES: list[str] = [
        'danceability', 'energy', 'loudness', 'speechiness',
        'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
    ]

    def __init__(self, url: str) -> None:
        """
        Load the dataset from *url*, clean it, and store as self.df.

        Parameters
        ----------
        url : str
            Raw CSV URL for the Spotify TidyTuesday dataset.
        """
        raw = pd.read_csv(url)
        raw = raw.dropna(subset=['track_name', 'track_artist'])
        self.df = raw.drop_duplicates(subset='track_id', keep='first').reset_index(drop=True)
        print(f"[SpotifyAnalyzer] Loaded {len(self.df):,} unique tracks.")

    # ── Stage 1 ──────────────────────────────────────────────────────────────
    def summary_statistics(self) -> pd.DataFrame:
        """
        Compute mean, median, std, and IQR for every numeric column.

        Returns
        -------
        pd.DataFrame
            Summary table with shape (n_numeric_cols, 4), indexed by column name.
            IQR is computed via np.percentile — no scipy.
        """
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        records = []
        for col in numeric_cols:
            arr = self.df[col].dropna().to_numpy()
            q75, q25 = np.percentile(arr, [75, 25])
            records.append({'Column': col,
                            'Mean': np.mean(arr),
                            'Median': np.median(arr),
                            'Std': np.std(arr, ddof=1),
                            'IQR': q75 - q25})
        return pd.DataFrame(records).set_index('Column').round(4)

    # ── Stage 2 ──────────────────────────────────────────────────────────────
    def genre_analysis(self) -> dict[str, object]:
        """
        Compute per-genre distribution stats and identify the highest-variance genre.

        Returns
        -------
        dict
            Keys:
            - 'genre_stats' (pd.DataFrame): mean/std/median per genre for four features.
            - 'highest_variance_genre' (str): genre with highest track_popularity variance.
        """
        feats = ['track_popularity', 'danceability', 'energy', 'valence']
        stats = self.df.groupby('playlist_genre')[feats].agg(['mean', 'std', 'median'])
        stats.columns = ['_'.join(c) for c in stats.columns]
        top = self.df.groupby('playlist_genre')['track_popularity'].var().idxmax()
        return {'genre_stats': stats.round(4), 'highest_variance_genre': top}

    # ── Stage 3a ─────────────────────────────────────────────────────────────
    def normalise_features(self) -> np.ndarray:
        """
        Min-max normalise the nine audio features to [0, 1].

        Returns
        -------
        np.ndarray
            Normalised matrix of shape (n_tracks, 9). Computed with pure
            NumPy broadcasting — no loops, no sklearn.
        """
        X = self.df[self.AUDIO_FEATURES].to_numpy(dtype=np.float64)
        X_min, X_max = X.min(axis=0), X.max(axis=0)
        return (X - X_min) / (X_max - X_min)

    # ── Stage 3b ─────────────────────────────────────────────────────────────
    def correlation_analysis(self) -> dict[str, object]:
        """
        Compute the 9×9 audio feature correlation matrix and top pairs.

        Returns
        -------
        dict
            Keys:
            - 'corr_matrix' (pd.DataFrame): full 9×9 correlation table.
            - 'max_positive_pair' (tuple[str, str, float]): feature pair & r-value.
            - 'max_negative_pair' (tuple[str, str, float]): feature pair & r-value.
        """
        X_norm = self.normalise_features()
        corr   = np.corrcoef(X_norm.T)
        n      = len(self.AUDIO_FEATURES)
        idx    = np.triu_indices(n, k=1)
        vals   = corr[idx]

        pos = np.argmax(vals); neg = np.argmin(vals)
        return {
            'corr_matrix': pd.DataFrame(corr, index=self.AUDIO_FEATURES,
                                        columns=self.AUDIO_FEATURES).round(4),
            'max_positive_pair': (self.AUDIO_FEATURES[idx[0][pos]],
                                  self.AUDIO_FEATURES[idx[1][pos]],
                                  round(float(vals[pos]), 4)),
            'max_negative_pair': (self.AUDIO_FEATURES[idx[0][neg]],
                                  self.AUDIO_FEATURES[idx[1][neg]],
                                  round(float(vals[neg]), 4)),
        }

    # ── Bonus 1 ──────────────────────────────────────────────────────────────
    def artist_fingerprint(self, artist_name: str) -> np.ndarray:
        """
        Return the mean audio feature vector for the specified artist.

        Parameters
        ----------
        artist_name : str
            Artist name exactly as stored in 'track_artist'.

        Returns
        -------
        np.ndarray
            Shape (9,) mean feature vector.
        """
        subset = self.df[self.df['track_artist'] == artist_name]
        if subset.empty:
            raise ValueError(f"Artist '{artist_name}' not found.")
        return subset[self.AUDIO_FEATURES].mean().to_numpy(dtype=np.float64)

    # ── Report ────────────────────────────────────────────────────────────────
    def generate_report(self) -> dict[str, object]:
        """
        Compile all key insights into a serialisable dictionary.

        Returns
        -------
        dict
            Contains dataset shape, genre variance winner, correlation extremes,
            high-energy popularity delta, and genre pair distances.
            All values are JSON-serialisable (str, int, float, list).
        """
        genre_res  = self.genre_analysis()
        corr_res   = self.correlation_analysis()

        X      = self.df[self.AUDIO_FEATURES].to_numpy(dtype=np.float64)
        e_idx  = self.AUDIO_FEATURES.index('energy')
        e_arr  = X[:, e_idx]
        pop    = self.df['track_popularity'].to_numpy(dtype=np.float64)
        mu, s  = e_arr.mean(), e_arr.std(ddof=1)
        mask   = e_arr > (mu + s)

        # filtered tracks
        filt_mask = (
            (self.df['track_popularity'] > 70) &
            (self.df['danceability']     > 0.7) &
            (self.df['energy']           > 0.6) &
            (self.df['duration_ms']      < 240_000)
        )
        filt_df = self.df[filt_mask]
        dom_genre = filt_df['playlist_genre'].value_counts().idxmax() if len(filt_df) else 'N/A'

        return {
            'dataset': {
                'total_unique_tracks': int(len(self.df)),
                'genres': self.df['playlist_genre'].nunique(),
                'columns': int(self.df.shape[1]),
            },
            'genre_analysis': {
                'highest_variance_genre': genre_res['highest_variance_genre'],
            },
            'correlation': {
                'max_positive_pair': list(corr_res['max_positive_pair']),
                'max_negative_pair': list(corr_res['max_negative_pair']),
            },
            'high_energy_tracks': {
                'threshold': round(float(mu + s), 4),
                'count': int(mask.sum()),
                'mean_popularity_high_energy': round(float(pop[mask].mean()), 2),
                'mean_popularity_overall': round(float(pop.mean()), 2),
                'delta': round(float(pop[mask].mean() - pop.mean()), 2),
            },
            'filtered_hits': {
                'count': int(len(filt_df)),
                'dominant_genre': dom_genre,
            },
        }


# ── Demo: run from a single cell ──────────────────────────────────────────────
analyzer = SpotifyAnalyzer(URL)
report   = analyzer.generate_report()
print(json.dumps(report, indent=2))


[SpotifyAnalyzer] Loaded 28,352 unique tracks.
{
  "dataset": {
    "total_unique_tracks": 28352,
    "genres": 6,
    "columns": 23
  },
  "genre_analysis": {
    "highest_variance_genre": "pop"
  },
  "correlation": {
    "max_positive_pair": [
      "energy",
      "loudness",
      0.6822
    ],
    "max_negative_pair": [
      "energy",
      "acousticness",
      -0.5459
    ]
  },
  "high_energy_tracks": {
    "threshold": 0.8819,
    "count": 4852,
    "mean_popularity_high_energy": 34.04,
    "mean_popularity_overall": 39.34,
    "delta": -5.3
  },
  "filtered_hits": {
    "count": 579,
    "dominant_genre": "pop"
  }
}


---
## 🏁 End of Notebook

All stages completed:

| Stage | Status |
|---|---|
| 1 – Setup & Inspection | ✅ |
| 2 – Genre & Popularity Analysis | ✅ |
| 3 – NumPy Analysis | ✅ |
| 4 – Documentation & AI Reflection | ✅ |
| Bonus 1 – Artist Fingerprint | ✅ |
| Bonus 2 – Genre Cluster Profile | ✅ |
| Bonus 3 – SpotifyAnalyzer Class | ✅ |
